# CSI Windows Multi-task Sensing with Location A Distance

This notebook demonstrates how to fine-tune the CSI sensing model so that it jointly:

1. Estimates the probability that the environment is empty (``p_empty``).
2. Predicts the distance from Location A to the subject when a person is present (``dA_cm``).
3. Converts the distance into a soft percentage toward Location A (``pA``) using the Gaussian mapping described in the project specification.

The workflow below mirrors the training CLI but keeps everything inline so you can experiment with parameters directly in the notebook.

## 1. Imports and configuration

In [ ]:
import os
import json
import subprocess
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader

from csi_sensing.data import CSIDataset, compute_normalization_stats, stratified_split
from csi_sensing.evaluation import evaluate_model
from csi_sensing.model import CSISensingModel
from csi_sensing.utils import compute_location_a_percentage

SEED = 7
torch.manual_seed(SEED)
np.random.seed(SEED)

DATA_ROOT = Path('dummy_data')
MANIFEST_CSV = DATA_ROOT / 'manifest.csv'
OUT_DIR = Path('notebook_runs/multitask')
OUT_DIR.mkdir(parents=True, exist_ok=True)

SIGMA_CM = 200.0
LAMBDA_DIST = 0.5
BATCH_SIZE = 32
EPOCHS = 5
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', DEVICE)


## 2. Prepare or generate a dataset

In [ ]:
if not MANIFEST_CSV.exists():
    print('Manifest not found, generating a dummy dataset for demonstration...')
    subprocess.run([
        'python',
        'examples/create_dummy_dataset.py',
        '--out_dir', str(DATA_ROOT),
        '--num_samples', '120',
        '--seed', str(SEED)
    ], check=True)
else:
    print('Using existing dataset at', DATA_ROOT)

df = pd.read_csv(MANIFEST_CSV)
df.sample(5, random_state=SEED)


## 3. Build DataLoaders with normalization

In [ ]:
train_df, val_df, test_df = stratified_split(df, train_ratio=0.7, val_ratio=0.15, test_ratio=0.15, random_state=SEED)
print('Split sizes:', len(train_df), len(val_df), len(test_df))

max_range = float(df['distA_cm'].max())
train_raw = CSIDataset(train_df.itertuples(index=False), str(DATA_ROOT), normalization=None, max_range_cm=max_range)
stats = compute_normalization_stats(train_raw)
stats_path = OUT_DIR / 'scaler.pkl'
stats.save(str(stats_path))
print('Saved normalization stats to', stats_path)

train_ds = CSIDataset(train_df.itertuples(index=False), str(DATA_ROOT), stats, max_range_cm=max_range)
val_ds = CSIDataset(val_df.itertuples(index=False), str(DATA_ROOT), stats, max_range_cm=max_range)
test_ds = CSIDataset(test_df.itertuples(index=False), str(DATA_ROOT), stats, max_range_cm=max_range)

def collate_fn(batch):
    return {
        'window': torch.stack([item['window'] for item in batch]),
        'label_empty': torch.stack([item['label_empty'] for item in batch]),
        'distA_cm': torch.stack([item['distA_cm'] for item in batch]),
    }

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=0, collate_fn=collate_fn)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0, collate_fn=collate_fn)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0, collate_fn=collate_fn)


## 4. Define the multi-task training step

In [ ]:
bce_loss = nn.BCEWithLogitsLoss()
smooth_l1 = nn.SmoothL1Loss(reduction='none')

def train_one_epoch(model, dataloader, optimizer):
    model.train()
    totals = {'loss': 0.0, 'loss_empty': 0.0, 'loss_dist': 0.0}
    batches = 0
    for batch in dataloader:
        optimizer.zero_grad(set_to_none=True)
        inputs = batch['window'].to(DEVICE)
        labels = batch['label_empty'].to(DEVICE)
        dists = batch['distA_cm'].to(DEVICE)
        mask_present = (labels == 0).float()

        logits, dist_pred = model(inputs)
        loss_empty = bce_loss(logits, labels)
        dist_loss_all = smooth_l1(dist_pred, dists)
        if mask_present.sum() > 0:
            dist_loss = (dist_loss_all * mask_present).sum() / mask_present.sum()
        else:
            dist_loss = torch.tensor(0.0, device=DEVICE)
        loss = loss_empty + LAMBDA_DIST * dist_loss
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        totals['loss'] += loss.item()
        totals['loss_empty'] += loss_empty.item()
        totals['loss_dist'] += dist_loss.item()
        batches += 1
    return {k: v / max(batches, 1) for k, v in totals.items()}


## 5. Train and monitor validation F1

In [ ]:
model = CSISensingModel().to(DEVICE)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)

best_state = None
best_f1 = -float('inf')
history = []
for epoch in range(1, EPOCHS + 1):
    train_metrics = train_one_epoch(model, train_loader, optimizer)
    val_metrics = evaluate_model(model, val_loader, DEVICE, sigma_cm=SIGMA_CM, threshold_empty=0.5)
    f1 = val_metrics['presence'].f1
    history.append({'epoch': epoch, 'train': train_metrics, 'val_f1': f1})
    if f1 > best_f1:
        best_f1 = f1
        best_state = {k: v.cpu() for k, v in model.state_dict().items()}
    print(f"Epoch {epoch:02d} | train_loss={train_metrics['loss']:.4f} | val_f1={f1:.4f}")

if best_state is not None:
    model.load_state_dict(best_state)
torch.save(model.state_dict(), OUT_DIR / 'best_model.pt')
print('Best validation F1:', best_f1)


## 6. Evaluate on the held-out test set

In [ ]:
metrics_test = evaluate_model(model, test_loader, DEVICE, sigma_cm=SIGMA_CM, threshold_empty=0.5)
presence = metrics_test['presence']
distance = metrics_test['distance']
print(f"Presence accuracy: {presence.accuracy:.3f} | F1: {presence.f1:.3f}")
print(f"Distance MAE (non-empty): {distance.mae:.2f} cm")
print(f"Mean Location-A percentage: {metrics_test['mean_pA']:.3f}")
print('Correlation between proximity and pA:', metrics_test['corr_proximity_pA'])


## 7. Run inference on a single sample

In [ ]:
sample = test_ds[0]
with torch.no_grad():
    window = sample['window'][None].to(DEVICE)
    logits, dist_pred = model(window)
    p_empty = torch.sigmoid(logits).item()
    dA_cm = dist_pred.item()

pA = compute_location_a_percentage(
    np.array([dA_cm]),
    sigma_cm=SIGMA_CM,
    empty_prob=np.array([p_empty]),
    threshold_empty=0.5
)[0]

print(json.dumps({
    'p_empty': p_empty,
    'dA_cm': dA_cm,
    'pA': pA,
    'label_empty': float(sample['label_empty'].item()),
    'true_dist_cm': float(sample['distA_cm'].item())
}, indent=2))
